In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers.legacy import Adam

In [2]:
TRAIN_PATH = "/Users/sabari/Documents/DS/GUVI/SmartVision-AI/smartvision_dataset/classification/train"
VAL_PATH = "/Users/sabari/Documents/DS/GUVI/SmartVision-AI/smartvision_dataset/classification/val"

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    VAL_PATH,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 1750 images belonging to 25 classes.
Found 375 images belonging to 25 classes.


In [3]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

2026-04-18 19:05:03.299056: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-18 19:05:03.299081: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-18 19:05:03.299092: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-04-18 19:05:03.299130: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-18 19:05:03.299145: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

outputs = Dense(train_generator.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

In [6]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [7]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25
)

Epoch 1/25


2026-04-18 19:05:06.331132: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


55/55 [==============================] - 16s 244ms/step - loss: 3.8726 - accuracy: 0.0451 - val_loss: 3.2558 - val_accuracy: 0.0400
Epoch 2/25
55/55 [==============================] - 11s 197ms/step - loss: 3.6017 - accuracy: 0.0640 - val_loss: 3.2774 - val_accuracy: 0.0400
Epoch 3/25
55/55 [==============================] - 12s 212ms/step - loss: 3.5277 - accuracy: 0.0634 - val_loss: 3.3550 - val_accuracy: 0.0400
Epoch 4/25
55/55 [==============================] - 11s 205ms/step - loss: 3.4170 - accuracy: 0.0726 - val_loss: 3.5103 - val_accuracy: 0.0400
Epoch 5/25
55/55 [==============================] - 11s 200ms/step - loss: 3.3766 - accuracy: 0.0869 - val_loss: 3.6512 - val_accuracy: 0.0400
Epoch 6/25
55/55 [==============================] - 11s 201ms/step - loss: 3.2879 - accuracy: 0.1017 - val_loss: 3.7033 - val_accuracy: 0.0400
Epoch 7/25
55/55 [==============================] - 11s 208ms/step - loss: 3.2720 - accuracy: 0.1166 - val_loss: 3.7411 - val_accuracy: 0.0240
Epoch 8/25

In [9]:
model.save("models/efficientnet_model.h5")
print("EfficientNet Model Saved ✅")

EfficientNet Model Saved ✅
